<a href="https://colab.research.google.com/github/reshmiee/codemixing_llmrouter/blob/main/dataset_reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q datasets

In [3]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset('JeremiahZ/hendrycks_math_merged', split='test')
m = ds.to_pandas()
print(len(m))                    # should be 5000
print(m['level'].value_counts()) # should show Level 1 to Level 5
print(m.columns.tolist())        # problem, level, type, solution

README.md:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.99MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.86MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

5000
level
Level 5    1324
Level 4    1214
Level 3    1131
Level 2     894
Level 1     437
Name: count, dtype: int64
['problem', 'level', 'type', 'solution']


In [4]:
m['lvl'] = m['level'].str.extract(r'(\d)')[0].astype(int)
m['source_id'] = ['MATH-test-' + str(i) for i in m.index]

plain = ~m['problem'].str.contains(r'\\|\$|\[asy\]', regex=True)
print('plain problems by level:', m[plain].groupby('lvl').size().to_dict())

def math_sample(mask, difficulty, label):
    d = m[plain & mask].sample(30, random_state=0)
    return pd.DataFrame({'task_type': 'Reasoning', 'difficulty': difficulty,
                         'source': label, 'source_id': d['source_id'].values,
                         'english': d['problem'].values})

mth = pd.concat([math_sample(m['lvl'] <= 2, 'Easy', 'MATH-L1-2'),
                 math_sample(m['lvl'] >= 4, 'Hard', 'MATH-L4-5')], ignore_index=True)
mth.to_csv('math_candidates.csv', index=False)
print(mth.groupby(['source', 'difficulty']).size())

plain problems by level: {1: 85, 2: 189, 3: 248, 4: 198, 5: 189}
source     difficulty
MATH-L1-2  Easy          30
MATH-L4-5  Hard          30
dtype: int64


In [6]:
from google.colab import files
files.download('math_candidates.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import pandas as pd

mth = pd.read_csv('math_candidates.csv')
CAP = 300  # placeholder, agree this with your team first

mth['too_long'] = mth['english'].str.len() > CAP
print(mth.groupby('difficulty')['too_long'].sum())   # how many dropped per cell

kept = mth[~mth['too_long']].drop(columns='too_long')
kept = kept.groupby('difficulty').head(19)           # first 19 per cell, in the seeded order
print(kept.groupby('difficulty').size())

kept.to_csv('math_trimmed.csv', index=False)

difficulty
Easy    1
Hard    3
Name: too_long, dtype: int64
difficulty
Easy    19
Hard    19
dtype: int64


In [8]:
from google.colab import files
files.download('math_trimmed.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import pandas as pd
from google.colab import files

d = pd.read_csv('math_trimmed.csv')

with open('english_problems.txt', 'w', encoding='utf-8') as f:
    for i, r in d.iterrows():
        f.write(f"{i+1}. [{r['difficulty']}] {r['english']}\n\n")

files.download('english_problems.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import pandas as pd, re
from google.colab import files

d = pd.read_csv('math_trimmed.csv')

text = """
1. [Easy] Catherine ke paas 6 gallon doodh hai aur woh 26/5 gallon cookies banane mein use karti hai. Uske paas kitna fraction gallon bacha?

2. [Easy] Asian elephant ka average gestation period 609 din hota hai. Yeh kitne weeks hai?

3. [Easy] Wendy ne notice kiya ki jab usne apne quarters ko 5 ke piles mein stack kiya toh 3 bache, aur jab 7 ke piles mein stack kiya toh 5 bache. Agar uske paas das dollar se kam ke quarters hain, toh uske paas kitne quarters hain?

4. [Easy] Teen consecutive integers ka product 720 hai. Inme se sabse bada integer kya hai?

5. [Easy] Ek right square pyramid ke base ke edges har ek 20 feet hain. Agar pyramid ki height 27 feet hai, toh uska volume cubic feet mein kya hai?

6. [Easy] Pascal's triangle ki us row mein jo 1 se shuru hoti hai aur phir 10 aata hai, next number kya hai?

7. [Easy] Ek regular hexagon ko chhe congruent equilateral triangles mein divide kiya gaya hai. Agar inme se ek triangle ka perimeter 39 inches hai, toh regular hexagon ka perimeter inches mein kya hai?

8. [Easy] Do numbers ka sum 25 hai aur unka difference 11 hai. Inme se chhota number kya hai?

9. [Easy] Sabse chhota three-digit positive integer kya hai jo 1 (mod 7) congruent ho?

10. [Easy] Ek figure skater north ki taraf face kar rahi hai jab woh apne right mein spin karna shuru karti hai. Woh 2250 degrees spin karti hai. Spin khatam hone par woh kis direction (north, south, east ya west) mein face kar rahi hogi?

11. [Easy] 8, 12, 16, 20 aur 24 ka mean kya hai?

12. [Easy] Agar ek 12-hour analog clock 8:00 dikha raha hai, toh minute aur hour hands ke beech ka smaller angle kitne degrees ka hai?

13. [Easy] Total four-digit palindromes ki sankhya find karo. (Yaad rahe ki palindrome ek nonnegative digit sequence hai jo aage aur peeche dono taraf same padhi jaati hai, jaise 1331. Zero first digit nahi ho sakta.)

14. [Easy] Maan lo mere paas 6 alag-alag books hain, jinme se 2 math books hain. Main apni 6 books shelf par kitne tarikon se stack kar sakta hoon agar mujhe stack ke dono ends par math book chahiye?

15. [Easy] Paanch consecutive integers ka sum paanch hai. Paanch integers ka product kya hai?

16. [Easy] 64 ke positive square root aur 64 ke cube root ke beech ka difference kya hai?

17. [Easy] Ek isosceles trapezoid ki sides 4 cm, 10 cm, 16 cm aur 10 cm hain. Is trapezoid ka area square centimeters mein kya hai?

18. [Easy] Hamare club mein 20 members hain, 10 boys aur 10 girls. Hum president aur vice-president kitne tarikon se choose kar sakte hain agar koi restriction nahi ho siwaye iske ki same person dono offices nahi hold kar sakta?

19. [Easy] Kaunsa positive two-digit integer apne digits ke sum ka exactly double hai?

20. [Hard] Labels dekhe bina, Adrien ne chaar CDs chaar cases mein rakhi. Probability kya hai ki exactly do CDs galat cases mein hain? Apna answer common fraction mein express karo.

21. [Hard] Word `SIXTEEN' ke letters randomly arrange kiye gaye hain. Probability kya hai ki dono E's ek dusre ke next nahi hain?

22. [Hard] 1 se 100 tak ke sabhi integers ki list mein digit 6 kitni baar aata hai?

23. [Hard] Ek right pyramid ka square base hai jiska area 288 square cm hai. Uska peak baaki sabhi vertices se 15 cm door hai. Pyramid ka volume cubic centimeters mein kya hai?

24. [Hard] 100 items ke true-false test mein, har question jo 4 ka multiple hai woh true hai, aur baaki sab false hain. Agar ek student har item jo 3 ka multiple hai usko false mark karta hai aur baaki sab ko true, toh 100 items mein se kitne correctly answer honge?

25. [Hard] Hamare club mein 25 members hain, aur hum president, secretary, aur treasurer pick karna chahte hain. Hum officers kitne tarikon se choose kar sakte hain, agar individual members ek se zyada office hold kar sakte hain?

26. [Hard] 25 students ka ek test par average 84 hai. 20 students ke another group ka average 66 hai. Overall average find karo.

27. [Hard] Ek textbook mein 1,000 pages hain. Kitne pages ke page numbers ke digits ka sum exactly 4 hota hai?

28. [Hard] Mere drawer mein 18 distinguishable socks hain: 8 white, 6 brown, aur 4 blue. Main kitne tarikon se ek pair of socks choose kar sakta hoon, provided ki mujhe same color ke do socks milen?

29. [Hard] Mere school ke math club mein 6 boys aur 8 girls hain. Mujhe state math competition ke liye ek team select karni hai. Humein team mein 6 log chahiye. Main team kitne tarikon se select kar sakta hoon taaki girls boys se zyada hon?

30. [Hard] Maria ke paas teen identical apples aur teen identical oranges hain. Woh fruits apne chaar friends mein kitne tarikon se distribute kar sakti hai agar woh Jacky ko koi orange nahi deti? (Note: Maria khud koi fruit nahi leti, aur kuch friends ko koi fruit nahi mil sakta.)

31. [Hard] Ek particular convex pentagon mein do congruent, acute angles hain. Baaki har interior angle ka measure un do acute angles ke measures ke sum ke barabar hai. Large angles ka common measure degrees mein kya hai?

32. [Hard] 99 aur 999 ke beech kitne whole numbers mein exactly ek 0 aata hai?

33. [Hard] Ek Senate committee mein 5 Democrats, 5 Republicans, aur 1 Independent hain. Woh ek circular table ke around kitne tarikon se baith sakte hain agar har party ke sabhi members ek dusre ke next baithein? (Do seatings equivalent maani jaati hain agar ek dusre ka rotation ho.)

34. [Hard] Ek Senate committee mein 8 Republicans aur 6 Democrats hain. Hum 5 members ki subcommittee kitne tarikon se bana sakte hain jisme har party se kam se kam ek member ho?

35. [Hard] 2008 se kam kitne positive even perfect cubes hain?

36. [Hard] Das log ek round table ke around baithe hain. Unme se teen ko randomly presentation dene ke liye choose kiya jaata hai. Probability kya hai ki teen chosen log consecutive seats par baithe the?

37. [Hard] Ek right triangle diya gaya hai jiski side lengths sab 8 ke integer multiples hain. Aise triangle ka smallest possible perimeter kitne units ka hoga?

38. [Hard] Kathy ko apni Europe trip ke liye paisa chahiye. Agar uske bank mein 300 US dollars hain lekin woh uski aadhi British pounds mein aur aadhi euros mein withdraw karna chahti hai, toh uske paas pounds se kitne zyada euros honge? Maan lo 1 pound = 1.64 USD aur 1 euro = 1.32 USD, aur nearest whole number mein round karo.
"""

items = {}
for line in text.splitlines():
    m = re.match(r'\s*(\d+)\.\s*\[(Easy|Hard)\]\s*(.+)', line)
    if m:
        items[int(m.group(1))] = (m.group(2), m.group(3).strip())

print('lines found:', len(items), '| rows in file:', len(d))
assert len(items) == len(d), 'count mismatch, check the pasted text'

d['hinglish'] = [items[i + 1][1] for i in range(len(d))]
bad = [i + 1 for i in range(len(d)) if items[i + 1][0] != d.loc[i, 'difficulty']]
print('rows where Easy/Hard label does not match:', bad)

d.to_csv('math_with_hinglish.csv', index=False, encoding='utf-8')
files.download('math_with_hinglish.csv')

lines found: 38 | rows in file: 38
rows where Easy/Hard label does not match: []


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>